Listy chromosome i mRNA utworzone! Pierwszym krokiem będzie projektowanie starterów dla chromosome, a mRNA wstrzymam się do opracowania 

In [1]:
import pandas as pd
import numpy as np
import csv
import matplotlib as mpl 
import matplotlib.pyplot as plt

### deseq OUT

In [4]:
# podejcie MW #
#--------------#
# wyniki deseq:
# pvalue jest normalizowane do padj przez "the Holm method" http://www.math.ucla.edu/~anderson/rw1001/library/base/html/p.adjust.html
# w analizie odciecie jest przy padj = 0.001 (mozliwe bylo w DEBROWSER) i taki mam plik wynikowy, podzielony na UP/DOWN
# (w zasadzie dwa pliki), z ktorych na podstawie najwyzszego log2foldchange wzialem po 100 najwyzszch sekwencji,
# polaczylem razem i zblastowalem. Plik ktory jest ponizej to zlepek 100UP i 100DOWN wynikow deseq, tych samych, ktore sie blastowaly

In [2]:
df_deseq_out = pd.read_csv('/mnt/sdb1/Project_Sieja_MCLR_52/2_pipeline/7_counting_transcripts/noncoding_novel/opracowane_deseq/podejcie_MW/1d/PBS1d_vs_MC1d_DESEQ_100UP_100DOWN_together.csv', sep='\t')
df_deseq_out.head()

,ID,PBS_1d_1,PBS_1d_2,PBS_1d_3,PBS_1d_4,MC_1d_1,MC_1d_2,MC_1d_3,padj,log2FoldChange,pvalue,foldChange,log10padj
0,TRINITY_DN96009_c1_g3_i3,0.0,0.0,0.0,0.000000,1926.746939,838.569902,912.024085,1.819644e-43,12.791799,2.678902e-47,7091.127494,42.740014
1,TRINITY_DN96009_c1_g7_i1,0.0,0.0,0.0,0.000000,1545.529985,939.097763,954.080531,2.081684e-60,12.695019,1.094528e-64,6631.036435,59.681585
2,TRINITY_DN96413_c4_g3_i1,0.0,0.0,0.0,0.000000,1395.729284,899.724351,217.491910,4.846018e-22,12.242750,5.707493e-25,4846.575702,21.314615
3,TRINITY_DN80414_c8_g1_i6,0.0,0.0,0.0,0.000000,373.985197,226.187686,1210.024049,2.914640e-20,11.768755,4.260318e-23,3489.378944,19.535415
4,TRINITY_DN91051_c3_g1_i1,0.0,0.0,0.0,0.905415,1429.821857,965.067460,561.153159,7.112150e-42,11.755678,1.271429e-45,3457.893280,41.147999


# filtrowanie pliku wynikowego blasta

In [3]:
df_blast_results = pd.read_csv('/mnt/sdb1/Project_Sieja_MCLR_52/2_pipeline/7_counting_transcripts/noncoding_novel/opracowane_deseq/podejcie_MW/1d/salmo_salar_only/blast_salmo_ony.csv', sep='\t', low_memory=False)
df_blast_results.shape

(5205, 5)

### odfiltrowanie zbyt niskiego coverage

In [4]:
df_blast_results.columns

Index(['query', 'al_title', 'al_accession', 'evalue', 'coverage'], dtype='object')

In [4]:
df_blast_results_20cov = df_blast_results[df_blast_results['coverage'] > 20]
df_blast_results_20cov.head()

,query,al_title,al_accession,evalue,coverage
1,TRINITY_DN96009_c1_g7_i1,gi|929075892|ref|XM_014192771.1| PREDICTED: Sa...,XM_014192771,8.331800e-27,20.718232
2,TRINITY_DN96009_c1_g7_i1,gi|929075890|ref|XM_014192770.1| PREDICTED: Sa...,XM_014192770,8.331800e-27,20.718232
3,TRINITY_DN96009_c1_g7_i1,gi|185134006|ref|NM_001123570.1| Salmo salar i...,NM_001123570,8.331800e-27,20.718232
4,TRINITY_DN91919_c2_g1_i1,gi|929293431|ref|XM_014168489.1| PREDICTED: Sa...,XM_014168489,8.099330e-47,47.592068
5,TRINITY_DN91919_c2_g1_i1,gi|929293431|ref|XM_014168489.1| PREDICTED: Sa...,XM_014168489,2.996650e-26,25.212465


In [5]:
df_blast_results_20cov.shape

(1913, 5)

### filtrowanie mRNA only

In [4]:
# select best hit based on evalue (drop_duplicates)
df_best_hit = df_blast_results.sort_values('evalue', ascending=True).drop_duplicates('query').sort_index()


In [5]:
df_best_hit.shape

(130, 5)

In [6]:
# select what you need based on string

# w tym wypadku mRNA only oznacza pominiecie blastów z ncRNA, mixRNA i innych dziwnych, nie oznacza natomiast
# zabawy z chromosome i filtrowaniem tego dodatkowo, poniewaz blastujac tylko do salmo salar, nie mam (chyba)
# zadnych hitow chromosome. Chromosome (czyli hity bez mRNA) w tym wypadku to takie, ktore w blascie mialy NO HIT.
# czyli zamiast filtrowania po chromosome nalezy wrzucic w te grupe trinity, ktore mialy NO HIT.

df_mRNA_only = df_best_hit[df_best_hit['al_title'].str.contains('mRNA')]
df_mRNA_only.shape


(117, 5)

In [8]:
df_mRNA_only.to_csv('/mnt/sdb1/Project_Sieja_MCLR_52/2_pipeline/7_counting_transcripts/noncoding_novel/opracowane_deseq/podejcie_MW/1d/salmo_salar_only/PBS1d_vs_MCLR1d_mRNA_only.csv', index=False)

In [9]:
df_best_hit.sort_values(by='coverage', ascending=False, inplace=True)

In [10]:
# save df as file
df_best_hit.to_csv('/mnt/sdb1/Project_Sieja_MCLR_52/2_pipeline/7_counting_transcripts/noncoding_novel/opracowane_deseq/podejcie_MW/1d/salmo_salar_only/PBS1d_vs_MCLR1d_best_hit_wo_cov_filt.csv', index=False)

In [12]:
# łączenie odfiltrowanych mRNA z wynikami deseq

df_best_hit.rename(columns={'query' : 'ID'}, inplace=True)
df_best_hit.head()

,ID,al_title,al_accession,evalue,coverage
5148,TRINITY_DN88224_c4_g3_i1,gi|929293843|ref|XM_014168709.1| PREDICTED: Sa...,XM_014168709,1.500500e-153,106.818182
3620,TRINITY_DN92300_c3_g2_i8,gi|929259163|ref|XM_014150515.1| PREDICTED: Sa...,XM_014150515,1.070190e-30,103.555556
1083,TRINITY_DN102696_c3_g1_i2,gi|929269569|ref|XM_014155947.1| PREDICTED: Sa...,XM_014155947,2.996650e-26,102.973978
3400,TRINITY_DN101886_c4_g3_i1,gi|929232276|ref|XM_014137207.1| PREDICTED: Sa...,XM_014137207,1.576680e-118,102.150538
1255,TRINITY_DN112554_c1_g7_i1,gi|929293431|ref|XM_014168489.1| PREDICTED: Sa...,XM_014168489,3.486130e-100,100.168634


In [13]:
result = pd.merge(df_deseq_out, df_best_hit, on='ID')
result.sort_values(by='log2FoldChange', inplace=True)

In [14]:
result.to_csv('/mnt/sdb1/Project_Sieja_MCLR_52/2_pipeline/7_counting_transcripts/noncoding_novel/opracowane_deseq/podejcie_MW/1d/salmo_salar_only/PBS_1d_vs_MC2_1d_deseq_concat_with_best_hit.csv', index=False)

### filtrowanie NO HIT
#### filtrowanie chromosomes TUTAJ BEZ TEGO. chromosome poprzednio == no hit teraz

### czyli reszta: all - mRNA

1. w pliku wynikowym blasta nie ma zaznaczonych trinity, które nie miały hitu.
2. w tym wypadku: porownac liste trinity z wynikowej blasta 
3. z drugiej strony w zblastowanych, oprocz mRNA sa tez inne hity, jak ncRNA, miscRNA i podobne.
4. dlatego muszez zrobic dwa filtrowania:
   - wszystkie trinity porownac z mRNA
   - dolaczyc do nich hity po blascie nie bedace mRNA





In [14]:


non_mRNA = df_best_hit[~df_best_hit['al_title'].str.contains('mRNA')]

non_mRNA_best_hit = non_mRNA.sort_values('evalue', ascending=True).drop_duplicates('query').sort_index()

In [11]:
non_mRNA_best_hit.to_csv('/mnt/sdb1/Project_Sieja_MCLR_52/2_pipeline/7_counting_transcripts/noncoding_novel/opracowane_deseq/podejcie_MW/1d/salmo_salar_only/test.csv')

In [15]:
non_mRNA_best_hit.shape

(13, 5)

In [7]:
all_trinity_to_blast = pd.read_csv('/mnt/sdb1/Project_Sieja_MCLR_52/2_pipeline/7_counting_transcripts/noncoding_novel/opracowane_deseq/podejcie_MW/1d/100_UP_100_DOWN_best_fold_change.headers')

In [8]:
all_trinity_to_blast.shape

(200, 1)

In [9]:
no_hit = all_trinity_to_blast[~all_trinity_to_blast['id'].isin(df_best_hit['query'])]

In [10]:
no_hit.shape

(70, 1)

In [13]:
no_hit.head()

,id
2,TRINITY_DN96413_c4_g3_i1
3,TRINITY_DN80414_c8_g1_i6
4,TRINITY_DN91051_c3_g1_i1
7,TRINITY_DN91051_c3_g2_i1
8,TRINITY_DN106172_c0_g1_i1
